# Popularity baseline

Issue: #17

## Question

How well does 'show the most popular items in the category' do? This sets the lowest bar that every other method must clear.

## Data used

Inputs: `data/sample/events.parquet` (training window w0, days 0-13), `data/categories/item_category.parquet` (#8), `data/eval/ranking_queries.parquet` (#10).

Output: `data/results/popularity.parquet` (one row per evaluated session), used by the comparison in #29. Code: `notebooks/evalset.py` (shared rules), `metrics.py`, `bootstrap.py`.

**Popularity** of an item = its number of **click** events in the training window. Ties are broken by item ID (smaller first), so the ranking is deterministic.

## Evaluation rules used for every method

1. A method returns a ranked list of items **inside the query category**. Items already in the session prefix are **not allowed** in the list.
2. Relevant labels = held-out events on items that are in the query category and not in the prefix. (Labels outside the category cannot be found by a method that ranks inside it.)
3. Queries left with no relevant label are dropped, for **all** methods, so results stay comparable.
4. NDCG@10 uses gains click 1, cart 2, order 3. Recall and OTTO weighted recall are at 20.

In [ ]:
import resource
import sys
from pathlib import Path

import numpy as np
import polars as pl

sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")
from evalset import TOP, evaluate, prepare_eval, summarize
from split import WINDOWS, day_to_ms

DATA = Path("data") if Path("data").exists() else Path("../data")
TRAIN_END = day_to_ms(WINDOWS["w0 initial (weeks 1-2)"]["train_end_day"])
OUT = DATA / "results"
OUT.mkdir(parents=True, exist_ok=True)

## Popularity from the training window only

In [ ]:
events = pl.scan_parquet(DATA / "sample" / "events.parquet")
clicks = events.filter((pl.col("type") == 0) & (pl.col("ts") < TRAIN_END))
pop = clicks.group_by("aid").agg(pl.len().alias("clicks")).collect()

# leakage check (see #12): the counts must equal a recount over events before the train end,
# and must differ from counts over all weeks.
recount = dict(events.filter((pl.col("type") == 0) & (pl.col("ts") < TRAIN_END)).group_by("aid").len().collect().iter_rows())
assert dict(zip(pop["aid"].to_list(), pop["clicks"].to_list())) == recount
all_weeks = dict(events.filter(pl.col("type") == 0).group_by("aid").len().collect().iter_rows())
assert all_weeks != recount
assert clicks.select(pl.col("ts").max()).collect()[0, 0] < TRAIN_END
print(f"popularity from {pop['clicks'].sum():,} training-window clicks on {pop.height:,} items")

categories = pl.read_parquet(DATA / "categories" / "item_category.parquet")
ranked = (
    categories.join(pop, on="aid", how="left").with_columns(pl.col("clicks").fill_null(0))
    .sort(["cluster", "clicks", "aid"], descending=[False, True, False])
)
ranked_by_cat = {c: part["aid"].to_numpy() for (c,), part in ranked.group_by("cluster", maintain_order=True)}
print("categories:", len(ranked_by_cat), " items per category: min", min(map(len, ranked_by_cat.values())), "max", max(map(len, ranked_by_cat.values())))

## The ranker

In [ ]:
def popularity_rank(q):
    """Most clicked items of the query category, skipping items already in the prefix."""
    out = []
    for aid in ranked_by_cat[q["category"]]:
        if aid not in q["prefix"]:
            out.append(int(aid))
            if len(out) == TOP:
                break
    return out


# toy check: category 0 has items 10 (5 clicks), 11 (5 clicks), 12 (1 click); prefix contains 10
toy_ranked = {0: np.array([10, 11, 12])}
def toy_rank(q):
    return [int(a) for a in toy_ranked[q["category"]] if a not in q["prefix"]][:TOP]
assert toy_rank({"category": 0, "prefix": {10}}) == [11, 12]
print("toy ranker check passes")

## Evaluate on the ranking queries

In [ ]:
queries = pl.read_parquet(DATA / "eval" / "ranking_queries.parquet")
prepared, prep_stats = prepare_eval(queries)
print(prep_stats)
results = evaluate(prepared, popularity_rank)
summary = summarize(results)
print(f"evaluated {results.height:,} sessions")
for k, v in summary.items():
    if isinstance(v, tuple):
        print(f"{k:>22}: {v[0]:.4f}  [{v[1]:.4f}, {v[2]:.4f}]")
    else:
        print(f"{k:>22}: {v:.4f}  (OTTO official aggregate, no interval)")

## Checks

In [ ]:
assert results.height == prep_stats["kept"]
assert (results["n_returned"] > 0).all()
assert (results["ndcg10"].is_between(0, 1)).all() and (results["recall20"].is_between(0, 1)).all()
# no item of any list was in the prefix: evaluate() asserts this for every session
# popularity uses training-window data only: asserted above
results.write_parquet(OUT / "popularity.parquet")
again = pl.read_parquet(OUT / "popularity.parquet")
assert again.height == results.height
print("saved", OUT / "popularity.parquet", again.columns)
print("short lists (fewer than 100 items):", int((results["n_returned"] < TOP).sum()))
print("peak memory GB:", round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e9, 2))

## Result

The table above is the floor: any learned method must beat these numbers with a bootstrap interval that excludes zero for the paired difference. The per-session results are saved for the comparison in #29.

Things to carry forward: queries are dropped when no relevant label remains after removing labels outside the category and in the prefix (count above), and the same drop applies to every later method.

## What I learned

To be written by the owner of the project.